In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Libraries loaded")

Libraries loaded


In [4]:
CLEAN_CSV       = '../data/movies_cleaned.csv'
COLLECTION_NAME = 'movie_overviews'
VECTOR_SIZE     = 384

print("Config set")

Config set


In [22]:
print("Step 1: Loading dataset...")

df = pd.read_csv(CLEAN_CSV)

# Use overview as our text field (committed in S4 Product Vision)
df = df[['title', 'director', 'actor', 'overview', 'tagline',
         'genres_list', 'imdb_rating', 'roi', 'revenue']].dropna()

# Overview -  one rich text field
df['raw_text'] = df['overview']
print(f"Total movies: {len(df)}")
print(f"Avg text length: {df['raw_text'].str.len().mean():.0f} chars")
print(f"\nSample text:\n{df['raw_text'].iloc[0][:200]}")

Step 1: Loading dataset...
Total movies: 3146
Avg text length: 275 chars

Sample text:
Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inc


In [23]:
print("Step 2: Loading SentenceTransformer model...")

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded ")
print(f"Embedding size: 384 dimensions")

Step 2: Loading SentenceTransformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded 
Embedding size: 384 dimensions


In [24]:
print("Step 3: Generating embeddings...")
print("This may take a few minutes...")

texts = df['raw_text'].tolist()
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

print(f"\nEmbeddings shape: {embeddings.shape}")
print("Embeddings generated")

Step 3: Generating embeddings...
This may take a few minutes...


Batches:   0%|          | 0/50 [00:00<?, ?it/s]


Embeddings shape: (3146, 384)
Embeddings generated


In [25]:
print("Step 4: Connecting to Qdrant...")

client = QdrantClient(host='localhost', port=6333)

# Delete collection if exists then recreate
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE
    )
)

print(f"Collection '{COLLECTION_NAME}' created")

Step 4: Connecting to Qdrant...
Collection 'movie_overviews' created


In [26]:
print("Step 5: Uploading embeddings to Qdrant...")

points = []
for i, (_, row) in enumerate(df.iterrows()):
    points.append(PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            'node_id'    : row['title'],
            'raw_text'   : row['raw_text'],
            'title'      : row['title'],
            'director'   : row['director'],
            'actor'      : row['actor'],
            'genre'      : row['genres_list'],
            'imdb_rating': row['imdb_rating'],
            'roi'        : row['roi'],
            'revenue'    : row['revenue']
        }
    ))

# Upload in batches of 100
batch_size = 100
for i in range(0, len(points), batch_size):
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points[i:i+batch_size]
    )

print(f"Uploaded {len(points)} embeddings to Qdrant")

Step 5: Uploading embeddings to Qdrant...
Uploaded 3146 embeddings to Qdrant


In [27]:
print("Step 6: Running similarity search...")
print("Query: 'A director and actor who repeatedly collaborate on action blockbusters'")
print("-" * 70)

query_text = "two heroes fight together to save the world from destruction in epic battle"
query_vector = model.encode(query_text).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True
)

print(f"\nTop 5 Similar Movies:\n")
for i, hit in enumerate(results.points):
    print(f"{i+1}. {hit.payload['title']}")
    print(f"   Director: {hit.payload['director']} | Actor: {hit.payload['actor']}")
    print(f"   Genre: {hit.payload['genre']}")
    print(f"   IMDB: {hit.payload['imdb_rating']} | ROI: {hit.payload['roi']}%")
    print(f"   Score: {hit.score:.4f}")
    print(f"   Text: {hit.payload['raw_text'][:100]}...")
    print()

Step 6: Running similarity search...
Query: 'A director and actor who repeatedly collaborate on action blockbusters'
----------------------------------------------------------------------

Top 5 Similar Movies:

1. Godzilla vs. Kong
   Director: Andy Serkis | Actor: Paige O'Hara
   Genre: ['Action', 'Science Fiction', 'Thriller']
   IMDB: 8.0 | ROI: 135.06%
   Score: 0.4820
   Text: In a time when monsters walk the Earth, humanity’s fight for its future sets Godzilla and Kong on a ...

2. Jack the Giant Slayer
   Director: John Woo | Actor: Sacha Baron Cohen
   Genre: ['Fantasy', 'Action', 'Adventure', 'Family']
   IMDB: 7.3 | ROI: 1.38%
   Score: 0.4780
   Text: The story of an ancient war that is reignited when a young farmhand unwittingly opens a gateway betw...

3. Mortal Kombat
   Director: Steve McQueen | Actor: Julianna Margulies
   Genre: ['Action', 'Fantasy']
   IMDB: 5.5 | ROI: 578.87%
   Score: 0.4778
   Text: For nine generations an evil sorcerer has been victorious in hand

In [28]:
print("Query: Film curator use case from Product Vision")
print("-" * 70)

# This is the exact user journey from our S4 Product Vision
query_text = "two strangers stranded in space must trust each other to survive"
query_vector = model.encode(query_text).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True
)

print(f"Query: '{query_text}'\n")
print(f"Top 5 Similar Movies:\n")
for i, hit in enumerate(results.points):
    print(f"{i+1}. {hit.payload['title']}")
    print(f"   Director: {hit.payload['director']} | Actor: {hit.payload['actor']}")
    print(f"   Score: {hit.score:.4f}")
    print(f"   Text: {hit.payload['raw_text'][:120]}...")
    print()

Query: Film curator use case from Product Vision
----------------------------------------------------------------------
Query: 'two strangers stranded in space must trust each other to survive'

Top 5 Similar Movies:

1. The Mountain Between Us
   Director: Penny Marshall | Actor: Bill Murray
   Score: 0.6159
   Text: Stranded on a mountain after a tragic plane crash, two strangers must work together to endure the extreme elements of th...

2. Knight and Day
   Director: Ridley Scott | Actor: Anthony Hopkins
   Score: 0.5415
   Text: A fugitive couple goes on a glamorous and sometimes deadly adventure where nothing and no one – even themselves – are wh...

3. The Requin
   Director: Alex Kendrick | Actor: Alex Kendrick
   Score: 0.5000
   Text: A couple on a romantic getaway find themselves stranded at sea when a tropical storm sweeps away their villa. In order t...

4. Chaos Walking
   Director: Nicholas Stoller, Doug Sweetland | Actor: Scott Speedman
   Score: 0.4975
   Text: Two unl

In [29]:
print("Query: High ROI collaboration with Sci-Fi theme (with Qdrant filter)")
print("-" * 70)

from qdrant_client.models import Filter, FieldCondition, Range

query_text = "science fiction space adventure with stunning visual effects"
query_vector = model.encode(query_text).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True,
    query_filter=Filter(
        must=[
            FieldCondition(
                key='roi',
                range=Range(gte=200)  # only high ROI movies
            )
        ]
    )
)

print(f"Query: '{query_text}' (filtered: ROI >= 200%)\n")
print(f"Top 5 High ROI Sci-Fi Movies:\n")
for i, hit in enumerate(results.points):
    print(f"{i+1}. {hit.payload['title']}")
    print(f"   Director: {hit.payload['director']} | Actor: {hit.payload['actor']}")
    print(f"   ROI: {hit.payload['roi']}% | IMDB: {hit.payload['imdb_rating']}")
    print(f"   Score: {hit.score:.4f}")
    print()



Query: High ROI collaboration with Sci-Fi theme (with Qdrant filter)
----------------------------------------------------------------------
Query: 'science fiction space adventure with stunning visual effects' (filtered: ROI >= 200%)

Top 5 High ROI Sci-Fi Movies:

1. Interstellar
   Director: Christopher Nolan | Actor: Matthew McConaughey
   ROI: 325.29% | IMDB: 8.6
   Score: 0.3826

2. Earth
   Director: Paul Grimault | Actor: Jean Martin
   ROI: 626.67% | IMDB: 7.9
   Score: 0.3824

3. The Hills Have Eyes
   Director: Taylor Hackford | Actor: Mads Mikkelsen
   ROI: 364.16% | IMDB: 8.3
   Score: 0.3555

4. Fantastic Four
   Director: Niki Caro | Actor: Emile Hirsch
   ROI: 233.54% | IMDB: 8.1
   Score: 0.3443

5. Skyline
   Director: Jonathan Levine | Actor: Sandra Bullock
   ROI: 569.85% | IMDB: 6.7
   Score: 0.3436



In [33]:
print("R + A — Retrieve + Augment")
print("-" * 60)

query_text = "two strangers stranded in space must trust each other to survive"
query_vector = model.encode(query_text).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True
)

print(f"Query: '{query_text}'\n")
print("Retrieved + Augmented Results:\n")
for i, hit in enumerate(results.points):
    print(f"{i+1}. {hit.payload['title']}")
    print(f"   Similarity Score : {hit.score:.4f}")
    print(f"   Director + Actor : {hit.payload['director']} + {hit.payload['actor']}")
    print(f"   Genre            : {hit.payload['genre']}")
    print(f"   IMDB Rating      : {hit.payload['imdb_rating']}")
    print(f"   ROI              : {hit.payload['roi']}%")
    print(f"   Revenue          : ${hit.payload['revenue']:,.0f}")
    # Augment with business context
    roi = hit.payload['roi']
    if roi > 200:
        verdict = " High ROI — Strong investment candidate"
    elif roi > 0:
        verdict = " Moderate ROI — Proceed with caution"
    else:
        verdict = " Negative ROI — High risk"
    print(f"   Business Verdict : {verdict}")
    print()

print("Qdrant notebook complete")

R + A — Retrieve + Augment
------------------------------------------------------------
Query: 'two strangers stranded in space must trust each other to survive'

Retrieved + Augmented Results:

1. The Mountain Between Us
   Similarity Score : 0.6159
   Director + Actor : Penny Marshall + Bill Murray
   Genre            : ['Drama', 'Adventure', 'Romance']
   IMDB Rating      : 5.5
   ROI              : 79.52%
   Revenue          : $62,832,209
   Business Verdict :  Moderate ROI — Proceed with caution

2. Knight and Day
   Similarity Score : 0.5415
   Director + Actor : Ridley Scott + Anthony Hopkins
   Genre            : ['Action', 'Comedy']
   IMDB Rating      : 6.8
   ROI              : 123.92%
   Revenue          : $261,989,769
   Business Verdict :  Moderate ROI — Proceed with caution

3. The Requin
   Similarity Score : 0.5000
   Director + Actor : Alex Kendrick + Alex Kendrick
   Genre            : ['Thriller', 'Horror']
   IMDB Rating      : 7.0
   ROI              : -98.46%
   